# Fine-tune Qwen3-Coder for Java→Bedrock Translation

This notebook fine-tunes Qwen3-Coder-7B using QLoRA for Java Minecraft mod to Bedrock addon translation.

**Runtime**: Requires Google Colab with T4 GPU (free tier works!) or A100 for faster training.

**Estimated Cost**: ~$0-0.50 on Colab free tier (with T4)

Based on [Unsloth](https://unsloth.ai) QLoRA fine-tuning.

## 1. Setup & Installation

In [ ]:
# Install Unsloth, datasets, and other dependencies
!pip install --no-cache-dir unsloth fastapi uvicorn
!pip install --no-cache-dir xformers bitsandbytes huggingface_hub
!pip install --no-cache-dir datasets peft trl
!pip install --no-cache-dir accelerate sanic

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Model with QLoRA Configuration

In [ ]:
from unsloth import FastLanguageModel
import os

# Model configuration
max_seq_length = 2048  # Can increase for longer code
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4bit quantization to reduce memory

# Load Qwen3-Coder-7B with QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-Coder-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Add LoRA adapters for fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Rank - higher = more capacity, more memory
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Use Unsloth's gradient checkpointing
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

## 3. Prepare Training Data

Upload your `train.jsonl` file from the `fine_tuning_export.py` module.

In [ ]:
# Option 1: Upload from local (run this cell, then upload file)
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    print(f'Uploaded: {filename}')
    data_file = filename

In [ ]:
# Option 2: Download from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# data_file = '/content/drive/MyDrive/path/to/train.jsonl'

In [ ]:
# Option 3: Clone from GitHub/GitLab
# !git clone https://github.com/your/repo.git
# data_file = 'repo/path/to/train.jsonl'

In [ ]:
import json
from datasets import load_dataset

# Load dataset from JSONL
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

raw_data = load_jsonl(data_file)
print(f"Loaded {len(raw_data)} examples")

# Convert to HF dataset format
from datasets import Dataset

dataset = Dataset.from_list(raw_data)
print(f"Dataset: {dataset}")
print(f"Example format: {dataset[0]}")

## 4. Apply Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Use Qwen chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    messages = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize = False, 
            add_generation_prompt = True
        )
        for convo in messages
    ]
    return { "text" : texts }

dataset = dataset.map(
    formatting_prompts_func,
    batched = True,
    remove_columns=dataset.column_names
)

print(f"Formatted dataset: {dataset}")
print(f"Example text:\n{dataset[0]['text'][:500]}...")

## 5. Training Configuration

In [ ]:
from trl import SFTTrainingArguments
from unsloth import is_bf16_supported
from datetime import datetime

# Training arguments
training_args = SFTTrainingArguments(
    output_dir = f"./qwen_coder_finetuned_{datetime.now():%Y%m%d_%H%M%S}",
    
    # Model params
    num_train_epochs = 3,  # 1-3 recommended for fine-tuning
    per_device_train_batch_size = 4,  # Adjust based on GPU memory
    gradient_accumulation_steps = 2,  # Effective batch size = 4 * 2 = 8
    
    # Optimizer
    optim = "adamw_8bit",
    learning_rate = 2e-4,  # Standard for LoRA
    weight_decay = 0.01,
    
    # Scheduler
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.1,
    
    # Performance
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),
    max_grad_norm = 0.3,
    
    # Logging & Saving
    logging_steps = 10,
    save_strategy = "epoch",
    save_total_limit = 2,  # Keep only last 2 checkpoints
    
    # Misc
    seed = 42,
    model_max_padded_seq_length = max_seq_length,
)

## 6. Train the Model

In [ ]:
from unsloth import train_on_responses_only
from trl import SFTTrainer

# Create trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
)

# Only train on responses (not the prompt)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# Print memory stats
print("GPU memory before training:")
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
# Run training!
# This will take 1-3 hours on T4 GPU for 500 examples
trainer.train()

## 7. Save the Fine-tuned Model

In [ ]:
# Save the LoRA adapters
trainer.save_model("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("LoRA adapter saved to ./lora_adapter")

In [ ]:
# Optional: Push to HuggingFace Hub
# !pip install huggingface_hub
# from huggingface_hub import login
# login()  # Enter your HF token
# model.push_to_hub("your-username/qwen-coder-java-bedrock")
# tokenizer.push_to_hub("your-username/qwen-coder-java-bedrock")

## 8. Test the Fine-tuned Model

In [ ]:
# Reload model in inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_java_code = '''
package com.example.block;

import net.minecraft.block.Block;
import net.minecraft.block.material.Material;

public class ExampleBlock extends Block {
    public ExampleBlock() {
        super(Material.rock);
        setUnlocalizedName("example_block");
        setHardness(1.5f);
        setResistance(3.0f);
    }
}
'''

messages = [
    {"role": "system", "content": "You are an expert Java to Bedrock translator."},
    {"role": "user", "content": f"Translate this Java block to Bedrock addon format:\n```java\n{test_java_code}\n```"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    temperature = 0.2,
    do_sample = True,
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Bedrock output:")
print(result)

## 9. Export to Ollama (Optional)

In [ ]:
# Export to GGUF for Ollama
# !pip install unsloth
# 
# model.save_pretrained_gguf("./model", tokenizer)
# 
# Then use with Ollama:
# ollama create qwen-coder-java-bedrock -f ./model/Modelfile
# ollama run qwen-coder-java-bedrock

## Next Steps

1. **Upload LoRA adapter** to HuggingFace Hub
2. **Deploy** with Ollama, vLLM, or Unsloth Studio
3. **A/B test** against GPT-4o/Claude API
4. **Iterate** with more training data as conversions accumulate

See `ai-engine/docs/FINE_TUNING.md` for full documentation.